In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount("/content/drive")

print("\n==============================================")
print("SEARCHING FOR DATASET")
print("==============================================")

# Search common locations
search_paths = [
    "/content/drive/MyDrive/data_classification1.zip",
    "/content/drive/MyDrive/data_classification1",
]

found = []

for path in search_paths:
    if os.path.exists(path):
        found.append(path)

if found:
    print("✅ Found:")
    for path in found:
        print(path)
else:
    print("❌ Dataset not found in the expected locations.")

    print("\nMyDrive top-level items:")
    print(os.listdir("/content/drive/MyDrive")[:50])

print("==============================================")

Mounted at /content/drive

SEARCHING FOR DATASET
✅ Found:
/content/drive/MyDrive/data_classification1.zip


In [ ]:
# ============================================================
# CELL 3 — EXTRACT + VERIFY DATASET
# ============================================================

import os
import zipfile

ZIP_PATH = "/content/drive/MyDrive/data_classification1.zip"
EXTRACT_DIR = "/content"

# ------------------------------------------------------------
# Extract dataset
# ------------------------------------------------------------
print("Extracting dataset...")

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("✅ Extraction completed.")

# ------------------------------------------------------------
# Verify expected structure
# ------------------------------------------------------------
BASE_DIR = "/content/data_classification1"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR  = os.path.join(BASE_DIR, "test")

print("\n==============================================")
print("DATASET VERIFICATION")
print("==============================================")

print("Base directory :", BASE_DIR)
print("Train exists   :", os.path.exists(TRAIN_DIR))
print("Test exists    :", os.path.exists(TEST_DIR))

if os.path.exists(TRAIN_DIR):
    train_classes = sorted(os.listdir(TRAIN_DIR))
    print("Train classes  :", len(train_classes))

if os.path.exists(TEST_DIR):
    test_classes = sorted(os.listdir(TEST_DIR))
    print("Test classes   :", len(test_classes))

image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

train_count = 0
test_count = 0

for root, dirs, files in os.walk(TRAIN_DIR):
    train_count += sum(
        1 for f in files
        if f.lower().endswith(image_extensions)
    )

for root, dirs, files in os.walk(TEST_DIR):
    test_count += sum(
        1 for f in files
        if f.lower().endswith(image_extensions)
    )

print("\nTraining images :", train_count)
print("Test images     :", test_count)

print("==============================================")

Extracting dataset...
✅ Extraction completed.

DATASET VERIFICATION
Base directory : /content/data_classification1
Train exists   : True
Test exists    : True
Train classes  : 28
Test classes   : 28

Training images : 1927
Test images     : 409


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

SEED = 42

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 28

# Training set: 80%
# Validation set: 20%

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    color_mode="rgb",
    validation_split=0.20,
    subset="training",
    seed=SEED,
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    color_mode="rgb",
    validation_split=0.20,
    subset="validation",
    seed=SEED,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    color_mode="rgb",
    shuffle=False
)


normalization_layer = layers.Rescaling(1.0 / 255.0)

train_ds = train_ds.map(
    lambda x, y: (normalization_layer(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

val_ds = val_ds.map(
    lambda x, y: (normalization_layer(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

test_ds = test_ds.map(
    lambda x, y: (normalization_layer(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
)


print("\n==============================================")
print("MOBILENETV2 DATASET")
print("==============================================")

print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Classes:", NUM_CLASSES)

print(
    "Training batches:",
    tf.data.experimental.cardinality(train_ds).numpy()
)

print(
    "Validation batches:",
    tf.data.experimental.cardinality(val_ds).numpy()
)

print(
    "Test batches:",
    tf.data.experimental.cardinality(test_ds).numpy()
)

images, labels = next(iter(train_ds))

print("\nImage batch shape:", images.shape)
print("Label batch shape:", labels.shape)

print(
    "Image range:",
    float(tf.reduce_min(images).numpy()),
    "to",
    float(tf.reduce_max(images).numpy())
)

print("==============================================")

Found 1927 files belonging to 28 classes.
Using 1542 files for training.
Found 1927 files belonging to 28 classes.
Using 385 files for validation.
Found 409 files belonging to 28 classes.

MOBILENETV2 DATASET
Image size: (224, 224)
Batch size: 32
Classes: 28
Training batches: 49
Validation batches: 13
Test batches: 13

Image batch shape: (32, 224, 224, 3)
Label batch shape: (32, 28)
Image range: 0.0 to 1.0


In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.10),
    ],
    name="augmentation"
)


base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = True

# Freeze lower layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Keep last 30 layers trainable
for layer in base_model.layers[-30:]:
    layer.trainable = True


inputs = keras.Input(
    shape=(224, 224, 3),
    name="input_image"
)

x = data_augmentation(inputs)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D(
    name="global_average_pooling"
)(x)

x = layers.Dropout(
    0.3,
    name="dropout"
)(x)

x = layers.Dense(
    128,
    activation="relu",
    name="feature_dense"
)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    name="predictions"
)(x)

model = keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="MobileNetV2_TSR"
)


print("\n==============================================")
print("MOBILENETV2 ARCHITECTURE")
print("==============================================")

model.summary()

print("\nTotal parameters    :", model.count_params())

trainable_params = np.sum(
    [np.prod(v.shape) for v in model.trainable_weights]
)

non_trainable_params = np.sum(
    [np.prod(v.shape) for v in model.non_trainable_weights]
)

print("Trainable parameters:", trainable_params)
print("Non-trainable params:", non_trainable_params)

print("\nTrainable backbone layers:",
      sum(layer.trainable for layer in base_model.layers),
      "/",
      len(base_model.layers))

print("==============================================")

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step

MOBILENETV2 ARCHITECTURE


Model: "MobileNetV2_TSR"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling          │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_dense (Dense)           │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 28)             │         3,612 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,425,564 (9.25 MB)

 Trainable params: 1,693,980 (6.46 MB)

 Non-trainable params: 731,584 (2.79 MB)


Total parameters    : 2425564
Trainable parameters: 1693980
Non-trainable params: 731584

Trainable backbone layers: 30 / 154


In [ ]:
LEARNING_RATE = 0.001

optimizer = keras.optimizers.Adam(
    learning_rate=LEARNING_RATE
)

model.compile(
    optimizer=optimizer,
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("==============================================")
print("MOBILENETV2 COMPILED")
print("==============================================")

print("Optimizer       : Adam")
print("Learning Rate   :", LEARNING_RATE)
print("Loss            : categorical_crossentropy")
print("Metric          : accuracy")
print("Random Seed     :", SEED)

print("\nModel name:", model.name)
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)

print("==============================================")

MOBILENETV2 COMPILED
Optimizer       : Adam
Learning Rate   : 0.001
Loss            : categorical_crossentropy
Metric          : accuracy
Random Seed     : 42

Model name: MobileNetV2_TSR
Input shape: (None, 224, 224, 3)
Output shape: (None, 28)


In [ ]:
EPOCHS = 15

print("==============================================")
print("MOBILENETV2 TRAINING")
print("==============================================")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    verbose=1
)

print("\n==============================================")
print("TRAINING COMPLETED")
print("==============================================")

print("Total epochs completed:", len(history.history["loss"]))

print(
    "Final Training Accuracy :",
    history.history["accuracy"][-1]
)

print(
    "Final Validation Accuracy:",
    history.history["val_accuracy"][-1]
)

print(
    "Best Validation Accuracy :",
    max(history.history["val_accuracy"])
)

print(
    "Best Validation Epoch    :",
    np.argmax(history.history["val_accuracy"]) + 1
)

print("==============================================")

MOBILENETV2 TRAINING
Epoch 1/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 144s 3s/step - accuracy: 0.6200 - loss: 1.3643 - val_accuracy: 0.4312 - val_loss: 3.5100
Epoch 2/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 118s 2s/step - accuracy: 0.8353 - loss: 0.5337 - val_accuracy: 0.6260 - val_loss: 1.7230
Epoch 3/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 116s 2s/step - accuracy: 0.9066 - loss: 0.2894 - val_accuracy: 0.4753 - val_loss: 3.1031
Epoch 4/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 120s 2s/step - accuracy: 0.9034 - loss: 0.3434 - val_accuracy: 0.3558 - val_loss: 3.8521
Epoch 5/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 142s 2s/step - accuracy: 0.9183 - loss: 0.2497 - val_accuracy: 0.3610 - val_loss: 4.5891
Epoch 6/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 128s 3s/step - accuracy: 0.9423 - loss: 0.2163 - val_accuracy: 0.0260 - val_loss: 10.2112
Epoch 7/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 118s 2s/step - accuracy: 0.9384 - loss: 0.1804 - val_accuracy: 0.0545 - val_loss: 12.8882
Epoch 8/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 141s 2s/step - accuracy: 0.9604 - loss: 0.1211 - v

In [ ]:
base_model = keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze the entire backbone first
base_model.trainable = False

# Unfreeze only the top 10 backbone layers
for layer in base_model.layers[-10:]:
    layer.trainable = True


inputs = keras.Input(
    shape=(224, 224, 3),
    name="input_image"
)

x = data_augmentation(inputs)

# Keep BatchNorm layers in inference mode
x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D(
    name="global_average_pooling"
)(x)

x = layers.Dropout(
    0.3,
    name="dropout"
)(x)

x = layers.Dense(
    128,
    activation="relu",
    name="feature_dense"
)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    name="predictions"
)(x)

model = keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="MobileNetV2_TSR"
)


LEARNING_RATE = 1e-4

model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


trainable_params = np.sum(
    [np.prod(v.shape) for v in model.trainable_weights]
)

non_trainable_params = np.sum(
    [np.prod(v.shape) for v in model.non_trainable_weights]
)

print("==============================================")
print("MOBILENETV2 STABLE FINE-TUNING CONFIGURATION")
print("==============================================")

print("Learning rate:", LEARNING_RATE)

print(
    "Trainable backbone layers:",
    sum(layer.trainable for layer in base_model.layers),
    "/",
    len(base_model.layers)
)

print("Trainable parameters    :", trainable_params)
print("Non-trainable parameters:", non_trainable_params)

print("Input shape :", model.input_shape)
print("Output shape:", model.output_shape)

print("==============================================")

MOBILENETV2 STABLE FINE-TUNING CONFIGURATION
Learning rate: 0.0001
Trainable backbone layers: 10 / 154
Trainable parameters    : 900060
Non-trainable parameters: 1525504
Input shape : (None, 224, 224, 3)
Output shape: (None, 28)


In [ ]:
EPOCHS = 15

print("==============================================")
print("STABLE MOBILENETV2 TRAINING")
print("==============================================")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    verbose=1
)

print("\n==============================================")
print("TRAINING COMPLETED")
print("==============================================")

print("Total epochs completed:",
      len(history.history["loss"]))

print(
    "Final Training Accuracy :",
    history.history["accuracy"][-1]
)

print(
    "Final Validation Accuracy:",
    history.history["val_accuracy"][-1]
)

best_epoch = (
    np.argmax(history.history["val_accuracy"]) + 1
)

best_val_acc = max(
    history.history["val_accuracy"]
)

best_val_loss = min(
    history.history["val_loss"]
)

print(
    "Best Validation Accuracy :",
    best_val_acc
)

print(
    "Best Validation Epoch    :",
    best_epoch
)

print(
    "Minimum Validation Loss  :",
    best_val_loss
)

print("==============================================")

STABLE MOBILENETV2 TRAINING
Epoch 1/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 128s 2s/step - accuracy: 0.2724 - loss: 2.6805 - val_accuracy: 0.2390 - val_loss: 2.6818
Epoch 2/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 106s 2s/step - accuracy: 0.5668 - loss: 1.6033 - val_accuracy: 0.3948 - val_loss: 2.0029
Epoch 3/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 105s 2s/step - accuracy: 0.6790 - loss: 1.1566 - val_accuracy: 0.4883 - val_loss: 1.6868
Epoch 4/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 156s 2s/step - accuracy: 0.7419 - loss: 0.9198 - val_accuracy: 0.5013 - val_loss: 1.5710
Epoch 5/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 110s 2s/step - accuracy: 0.7892 - loss: 0.7500 - val_accuracy: 0.5195 - val_loss: 1.3840
Epoch 6/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 105s 2s/step - accuracy: 0.8223 - loss: 0.6085 - val_accuracy: 0.5688 - val_loss: 1.2544
Epoch 7/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 105s 2s/step - accuracy: 0.8567 - loss: 0.5145 - val_accuracy: 0.5896 - val_loss: 1.2347
Epoch 8/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 142s 2s/step - accuracy: 0.8684 - loss: 0.466

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("==============================================")
print("MOBILENETV2 FINAL TEST EVALUATION")
print("==============================================")


true_labels = []
pred_labels = []

for images, labels in test_ds:

    predictions = model.predict(
        images,
        verbose=0
    )

    true_labels.extend(
        np.argmax(labels.numpy(), axis=1)
    )

    pred_labels.extend(
        np.argmax(predictions, axis=1)
    )

true_labels = np.array(true_labels)
pred_labels = np.array(pred_labels)


accuracy = accuracy_score(
    true_labels,
    pred_labels
)

weighted_precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted",
    zero_division=0
)

weighted_recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted",
    zero_division=0
)

weighted_f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted",
    zero_division=0
)

print("\nTest Images          :", len(true_labels))
print("Accuracy             :", f"{accuracy * 100:.2f}%")
print("Weighted Precision   :", f"{weighted_precision:.4f}")
print("Weighted Recall      :", f"{weighted_recall:.4f}")
print("Weighted F1          :", f"{weighted_f1:.4f}")

print("==============================================")

MOBILENETV2 FINAL TEST EVALUATION

Test Images          : 409
Accuracy             : 64.06%
Weighted Precision   : 0.7962
Weighted Recall      : 0.6406
Weighted F1          : 0.6562


In [ ]:
import time

BATCH_SIZE_BENCH = 32
WARMUP_RUNS = 20
TIMED_RUNS = 100
DROP_FIRST = 10

benchmark_images, _ = next(iter(test_ds))

benchmark_images = benchmark_images[:BATCH_SIZE_BENCH]

print("==============================================")
print("MOBILENETV2 STRICT INFERENCE BENCHMARK")
print("==============================================")

print("Batch size:", benchmark_images.shape[0])
print("Warmup runs:", WARMUP_RUNS)
print("Timed runs:", TIMED_RUNS)
print("Dropped initial runs:", DROP_FIRST)

for _ in range(WARMUP_RUNS):
    _ = model(
        benchmark_images,
        training=False
    )

times = []

for _ in range(TIMED_RUNS):

    start_time = time.perf_counter()

    _ = model(
        benchmark_images,
        training=False
    )

    # Force TensorFlow synchronization
    _ = _.numpy()

    end_time = time.perf_counter()

    times.append(
        end_time - start_time
    )

times = np.array(times)

stable_times = times[DROP_FIRST:]

mean_batch_latency = np.mean(stable_times)
median_batch_latency = np.median(stable_times)

mean_per_image = (
    mean_batch_latency /
    benchmark_images.shape[0]
)

median_per_image = (
    median_batch_latency /
    benchmark_images.shape[0]
)

throughput = (
    benchmark_images.shape[0] /
    mean_batch_latency
)


print("\n==============================================")
print("FINAL LATENCY RESULTS")
print("==============================================")

print(
    "Mean batch latency   :",
    f"{mean_batch_latency:.6f} sec"
)

print(
    "Median batch latency :",
    f"{median_batch_latency:.6f} sec"
)

print(
    "Mean per-image time  :",
    f"{mean_per_image * 1000:.4f} ms"
)

print(
    "Median per-image     :",
    f"{median_per_image * 1000:.4f} ms"
)

print(
    "Throughput           :",
    f"{throughput:.2f} images/sec"
)

print("==============================================")

MOBILENETV2 STRICT INFERENCE BENCHMARK
Batch size: 32
Warmup runs: 20
Timed runs: 100
Dropped initial runs: 10

FINAL LATENCY RESULTS
Mean batch latency   : 2.303586 sec
Median batch latency : 2.014756 sec
Mean per-image time  : 71.9871 ms
Median per-image     : 62.9611 ms
Throughput           : 13.89 images/sec


In [ ]:
def make_gradcam_heatmap_mobilenet(
    img_array,
    model,
    base_model,
    pred_index=None
):
    """
    Standard Grad-CAM for MobileNetV2.

    Uses the last convolutional layer of the
    MobileNetV2 backbone.
    """

    last_conv_layer = None

    for layer in reversed(base_model.layers):

        try:
            output_shape = layer.output.shape

            if len(output_shape) == 4:
                last_conv_layer = layer
                break

        except:
            continue

    if last_conv_layer is None:
        raise ValueError(
            "Could not find a suitable 4D convolutional layer."
        )

    print("Grad-CAM layer:", last_conv_layer.name)

 
    grad_model = keras.models.Model(
        inputs=model.inputs,
        outputs=[
            last_conv_layer.output,
            model.output
        ]
    )

    with tf.GradientTape() as tape:

        conv_outputs, predictions = grad_model(
            img_array,
            training=False
        )

        if pred_index is None:
            pred_index = int(
                tf.argmax(
                    predictions[0]
                ).numpy()
            )
        else:
            pred_index = int(pred_index)

        class_channel = predictions[:, pred_index]

   
    grads = tape.gradient(
        class_channel,
        conv_outputs
    )

    pooled_grads = tf.reduce_mean(
        grads,
        axis=(0, 1, 2)
    )

    conv_outputs = conv_outputs[0]

    heatmap = tf.reduce_sum(
        conv_outputs * pooled_grads,
        axis=-1
    )


    heatmap = tf.maximum(
        heatmap,
        0
    )

    max_value = tf.reduce_max(heatmap)

    heatmap = tf.where(
        max_value > 0,
        heatmap / max_value,
        tf.zeros_like(heatmap)
    )

    heatmap = tf.image.resize(
        heatmap[..., tf.newaxis],
        (224, 224)
    )

    heatmap = tf.squeeze(
        heatmap,
        axis=-1
    )

    return (
        heatmap.numpy(),
        pred_index,
        last_conv_layer.name
    )

print("==============================================")
print("MOBILENETV2 GRAD-CAM FUNCTION READY")
print("==============================================")

MOBILENETV2 GRAD-CAM FUNCTION READY


In [ ]:
def make_gradcam_heatmap_mobilenet_fixed(
    img_array,
    model,
    base_model,
    pred_index=None
):
    """
    Fixed Grad-CAM implementation for the
    MobileNetV2 functional model.

    Uses the final convolutional feature map
    from the MobileNetV2 backbone.
    """


    last_conv_layer = base_model.get_layer("out_relu")

    classifier_input = keras.Input(
        shape=(7, 7, 1280),
        name="gradcam_feature_input"
    )

    x = model.get_layer(
        "global_average_pooling"
    )(classifier_input)

    x = model.get_layer(
        "dropout"
    )(x, training=False)

    x = model.get_layer(
        "feature_dense"
    )(x)

    classifier_output = model.get_layer(
        "predictions"
    )(x)

    classifier_model = keras.Model(
        classifier_input,
        classifier_output
    )

    feature_model = keras.Model(
        base_model.input,
        last_conv_layer.output
    )

    with tf.GradientTape() as tape:

        feature_maps = feature_model(
            img_array,
            training=False
        )

        predictions = classifier_model(
            feature_maps,
            training=False
        )

        if pred_index is None:
            pred_index = int(
                tf.argmax(
                    predictions[0]
                ).numpy()
            )
        else:
            pred_index = int(pred_index)

        class_channel = predictions[:, pred_index]


    grads = tape.gradient(
        class_channel,
        feature_maps
    )

    pooled_grads = tf.reduce_mean(
        grads,
        axis=(0, 1, 2)
    )

    feature_maps = feature_maps[0]

    heatmap = tf.reduce_sum(
        feature_maps * pooled_grads,
        axis=-1
    )

    heatmap = tf.maximum(
        heatmap,
        0
    )

   
    max_value = tf.reduce_max(heatmap)

    heatmap = tf.where(
        max_value > 0,
        heatmap / max_value,
        tf.zeros_like(heatmap)
    )


    heatmap = tf.image.resize(
        heatmap[..., tf.newaxis],
        (224, 224)
    )

    heatmap = tf.squeeze(
        heatmap,
        axis=-1
    )

    return (
        heatmap.numpy(),
        pred_index,
        last_conv_layer.name
    )


print("==============================================")
print("FIXED MOBILENETV2 GRAD-CAM FUNCTION READY")
print("==============================================")

FIXED MOBILENETV2 GRAD-CAM FUNCTION READY


In [ ]:
images, labels = next(iter(test_ds))

img = images[0]

img_array = tf.expand_dims(img, axis=0)

# True class
true_class = int(
    tf.argmax(labels[0]).numpy()
)

heatmap, predicted_class, gradcam_layer = (
    make_gradcam_heatmap_mobilenet_fixed(
        img_array,
        model,
        base_model
    )
)


important_pixels = np.sum(
    heatmap >= 0.5
)

total_pixels = heatmap.size

important_percentage = (
    important_pixels /
    total_pixels *
    100
)


print("=" * 60)
print("MOBILENETV2 GRAD-CAM SAMPLE VERIFICATION")
print("=" * 60)

print(f"True Class              : {true_class}")
print(f"Predicted Class         : {predicted_class}")
print(f"Grad-CAM Layer          : {gradcam_layer}")

print("-" * 60)

print(f"Heatmap Shape           : {heatmap.shape}")
print(f"Heatmap Min             : {heatmap.min():.6f}")
print(f"Heatmap Max             : {heatmap.max():.6f}")
print(f"Heatmap Mean            : {heatmap.mean():.6f}")
print(f"Heatmap Std             : {heatmap.std():.6f}")

print("-" * 60)

print(
    f"Important Pixels        : "
    f"{important_pixels}/{total_pixels}"
)

print(
    f"Important Pixel %       : "
    f"{important_percentage:.4f}%"
)

print("=" * 60)

MOBILENETV2 GRAD-CAM SAMPLE VERIFICATION
True Class              : 0
Predicted Class         : 0
Grad-CAM Layer          : out_relu
------------------------------------------------------------
Heatmap Shape           : (224, 224)
Heatmap Min             : 0.000000
Heatmap Max             : 0.995341
Heatmap Mean            : 0.296754
Heatmap Std             : 0.253412
------------------------------------------------------------
Important Pixels        : 11842/50176
Important Pixel %       : 23.6009%


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

sample_results = []

correct_predictions = 0
total_images = 0


for batch_images, batch_labels in test_ds:

    batch_size = batch_images.shape[0]

    for i in range(batch_size):

        img = batch_images[i]
        label = batch_labels[i]

        img_array = tf.expand_dims(img, axis=0)

        # True class
        true_class = int(
            tf.argmax(label).numpy()
        )

        # Grad-CAM
        heatmap, predicted_class, gradcam_layer = (
            make_gradcam_heatmap_mobilenet_fixed(
                img_array,
                model,
                base_model
            )
        )

  

        if predicted_class == true_class:
            correct_predictions += 1

        total_images += 1

        important_pixels = np.sum(
            heatmap >= 0.5
        )

        total_pixels = heatmap.size

        important_percentage = (
            important_pixels /
            total_pixels *
            100
        )

     
        average_heatmap_intensity = np.mean(
            heatmap
        )

        sample_results.append({
            "Image_Index": total_images - 1,
            "True_Class": true_class,
            "Predicted_Class": predicted_class,
            "Correct": int(
                predicted_class == true_class
            ),
            "Important_Pixels_Percentage":
                important_percentage,
            "Average_Heatmap_Intensity":
                average_heatmap_intensity
        })

    # Progress
    if total_images % 100 < batch_size:
        print(
            f"Processed {total_images}/{409} images"
        )

gradcam_df = pd.DataFrame(
    sample_results
)

mean_important_pixels = (
    gradcam_df[
        "Important_Pixels_Percentage"
    ].mean()
)

mean_heatmap_intensity = (
    gradcam_df[
        "Average_Heatmap_Intensity"
    ].mean()
)

prediction_accuracy = (
    correct_predictions /
    total_images *
    100
)

print("\n")
print("=" * 70)
print("MOBILENETV2 FULL TEST GRAD-CAM RESULTS")
print("=" * 70)

print(
    f"Total Test Images              : "
    f"{total_images}"
)

print(
    f"Prediction Accuracy            : "
    f"{prediction_accuracy:.2f}%"
)

print(
    f"Mean Important Pixels (%)      : "
    f"{mean_important_pixels:.4f}%"
)

print(
    f"Mean Average Heatmap Intensity : "
    f"{mean_heatmap_intensity:.4f}"
)

print(
    f"Grad-CAM Layer                 : "
    f"{gradcam_layer}"
)

print("=" * 70)

gradcam_df.to_csv(
    "MobileNetV2_GradCAM_Sample_Results.csv",
    index=False
)

print(
    "\nSaved: "
    "MobileNetV2_GradCAM_Sample_Results.csv"
)

Processed 128/409 images
Processed 224/409 images
Processed 320/409 images
Processed 409/409 images


MOBILENETV2 FULL TEST GRAD-CAM RESULTS
Total Test Images              : 409
Prediction Accuracy            : 64.06%
Mean Important Pixels (%)      : 20.6481%
Mean Average Heatmap Intensity : 0.2678
Grad-CAM Layer                 : out_relu

Saved: MobileNetV2_GradCAM_Sample_Results.csv


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

class_heatmaps = {
    class_id: []
    for class_id in range(28)
}

processed = 0

for batch_images, batch_labels in test_ds:

    batch_size = batch_images.shape[0]

    for i in range(batch_size):

        img = batch_images[i]
        label = batch_labels[i]

        img_array = tf.expand_dims(
            img,
            axis=0
        )

             true_class = int(
            tf.argmax(label).numpy()
        )

      
        heatmap, predicted_class, _ = (
            make_gradcam_heatmap_mobilenet_fixed(
                img_array,
                model,
                base_model
            )
        )

        heatmap_flat = heatmap.flatten()

        norm = np.linalg.norm(
            heatmap_flat
        )

        if norm > 0:
            heatmap_flat = (
                heatmap_flat / norm
            )

        class_heatmaps[
            true_class
        ].append(heatmap_flat)

        processed += 1

    if processed % 100 < batch_size:
        print(
            f"Processed {processed}/409 images"
        )

class_results = []

for class_id in range(28):

    heatmaps = np.array(
        class_heatmaps[class_id]
    )

    if len(heatmaps) == 0:
        consistency = np.nan

    elif len(heatmaps) == 1:
        consistency = 1.0

    else:

        mean_attention = np.mean(
            heatmaps,
            axis=0
        )

        mean_norm = np.linalg.norm(
            mean_attention
        )

        if mean_norm > 0:
            mean_attention = (
                mean_attention /
                mean_norm
            )

        similarities = np.dot(
            heatmaps,
            mean_attention
        )

        consistency = np.mean(
            similarities
        )

    class_results.append({
        "Class_ID": class_id,
        "Number_of_Images": len(
            class_heatmaps[class_id]
        ),
        "Attention_Consistency":
            consistency
    })

classwise_df = pd.DataFrame(
    class_results
)

overall_consistency = (
    classwise_df[
        "Attention_Consistency"
    ].mean()
)

print("\n")
print("=" * 70)
print("MOBILENETV2 CLASS-WISE ATTENTION CONSISTENCY")
print("=" * 70)

print(
    classwise_df.to_string(
        index=False
    )
)

print("-" * 70)

print(
    f"Overall Attention Consistency : "
    f"{overall_consistency:.4f}"
)

print("=" * 70)

classwise_df.to_csv(
    "MobileNetV2_Classwise_Attention_Consistency.csv",
    index=False
)

print(
    "\nSaved: "
    "MobileNetV2_Classwise_Attention_Consistency.csv"
)

Processed 128/409 images
Processed 224/409 images
Processed 320/409 images
Processed 409/409 images


MOBILENETV2 CLASS-WISE ATTENTION CONSISTENCY
 Class_ID  Number_of_Images  Attention_Consistency
        0                22               0.931358
        1                 2               0.922184
        2                11               0.979440
        3                 8               0.960646
        4                 3               0.897213
        5                 3               0.827171
        6                 3               0.956954
        7                26               0.799244
        8                 2               0.980027
        9                50               0.783138
       10                11               0.789692
       11                18               0.919958
       12                17               0.833292
       13                33               0.842824
       14                16               0.795998
       15                 3          

In [ ]:
mobilenetv2_final_summary = pd.DataFrame([{
    "Model": "MobileNetV2",

    "Accuracy (%)": 64.06,
    "Weighted Precision": 0.7962,
    "Weighted Recall": 0.6406,
    "Weighted F1": 0.6562,

    # Grad-CAM
    "Mean Important Pixels (%)":
        20.6481,
    "Mean Heatmap Intensity":
        0.2678,
    "Overall Attention Consistency":
        0.8836,
    "GradCAM Layer":
        "out_relu",

    # Inference
    "Mean Batch Latency (sec)":
        2.303586,
    "Median Batch Latency (sec)":
        2.014756,
    "Mean Inference Time per Image (ms)":
        71.9871,
    "Median Inference Time per Image (ms)":
        62.9611,
    "Throughput (images/sec)":
        13.89
}])


print("=" * 80)
print("MOBILENETV2 FINAL SUMMARY")
print("=" * 80)

print(
    mobilenetv2_final_summary.to_string(
        index=False
    )
)

print("=" * 80)

mobilenetv2_final_summary.to_csv(
    "MobileNetV2_Final_Summary.csv",
    index=False
)

print(
    "\nSaved: MobileNetV2_Final_Summary.csv"
)

MOBILENETV2 FINAL SUMMARY
      Model  Accuracy (%)  Weighted Precision  Weighted Recall  Weighted F1  Mean Important Pixels (%)  Mean Heatmap Intensity  Overall Attention Consistency GradCAM Layer  Mean Batch Latency (sec)  Median Batch Latency (sec)  Mean Inference Time per Image (ms)  Median Inference Time per Image (ms)  Throughput (images/sec)
MobileNetV2         64.06              0.7962           0.6406       0.6562                    20.6481                  0.2678                         0.8836      out_relu                  2.303586                    2.014756                             71.9871                               62.9611                    13.89

Saved: MobileNetV2_Final_Summary.csv
